In [6]:
from ipynb.fs.full.ScPipelineProto import *
import cfad
from pathlib import Path
import math

In [30]:
def plot_lfp_by_region(mean_clean, window, df, mouse, depths, trial_type):
    pag_indices = df.index[df['acronym'] == 'PAG'].tolist()
    
    if pag_indices: # Only proceed if the probe actually hit the PAG in this animal
        # 2. Extract the depth values specifically for the PAG channels
        pag_depths = [depths[i] for i in pag_indices]
        
        # 3. Calculate the empirical boundaries for this specific subject
        min_pag_depth = min(pag_depths)  # The most dorsal PAG channel (closest to surface)
        max_pag_depth = max(pag_depths)  # The most ventral PAG channel (deepest)
        total_pag_span = max_pag_depth - min_pag_depth
        
        # Divide the span into equal thirds
        # Note: Assuming smaller depth values are more dorsal.
        DORSAL_BOUNDARY = min_pag_depth + (total_pag_span / 3)
        LATERAL_BOUNDARY = min_pag_depth + (2 * total_pag_span / 3)
        
        # 4. Reassign the acronyms dynamically
        for idx in pag_indices:
            channel_depth = depths[idx]
            
            if channel_depth < DORSAL_BOUNDARY:
                df.at[idx, 'acronym'] = 'dPAG'
            elif channel_depth < LATERAL_BOUNDARY:
                df.at[idx, 'acronym'] = 'lPAG'
            else:
                df.at[idx, 'acronym'] = 'vlPAG'
    
    df.sort_values(by = ["rel_y"])
    # 1. Setup Data Alignment
    # Assuming 'mean_clean' is your [channels, samples] array 
    # and 'df' has the exact same number of rows, perfectly ordered to match the channels.
    unique_regions = df['acronym'].dropna().unique()[::-1]
    num_regions = len(unique_regions)
    
    # 2. Determine Dynamic Grid Layout
    # Creates a grid with a maximum of 3 columns
    ncols = min(4, num_regions)
    nrows = math.ceil(num_regions / ncols)
    
    # 3. Create the Figure and Axes Grid
    fig8, axes = plt.subplots(nrows=nrows, ncols=ncols, figsize=(8 * ncols, 6 * nrows), sharex=True, sharey=True)
    axes = np.atleast_1d(axes).flatten() # Flatten to a 1D array for easy iteration
    
    num_channels, num_samples = mean_clean.shape
    time_vector = np.linspace(window[0], window[1], num_samples)
    
    # 4. Global Colormap Setup
    # We use the global min/max depth so the color scale remains consistent across all subplots
    norm = Normalize(vmin=np.min(depths), vmax=np.max(depths))
    cmap = plt.get_cmap('winter') 
    
    # 5. Loop through Regions and Plot
    for idx, region in enumerate(unique_regions):
        ax = axes[idx]
        
        # Identify which channels belong to the current region
        # df.index retrieves the row numbers that match the condition
        channel_indices = df.index[df['acronym'] == region].tolist()
        
        for i in channel_indices:
            channel_depth = depths[i]
            color = cmap(norm(channel_depth))
            
            # Plot the specific channel on this region's subplot
            ax.plot(time_vector, mean_clean[i, :], color=color, alpha=0.7, linewidth=0.8)
        
        # 6. Formatting the Subplot
        ax.set_title(f'{region}', fontsize=14, fontweight='bold')
        
        # Stimulus markers for fear conditioning / visual paradigms
        ax.axvline(x=0, color='black', linestyle='--', linewidth=1.5, alpha=0.8)
        if trial_type == "Loom":
            ax.axvline(x=1.47, color='black', linestyle='--', linewidth = 1.5, alpha=0.8)
            ax.axvline(x=2.94, color='black', linestyle='--', linewidth = 1.5, alpha=0.8)
    
        # Only add y-labels to the leftmost column to reduce clutter
        if idx % ncols == 0:
            ax.set_ylabel('Voltage (\u00B5V)', fontsize=12)
        # Only add x-labels to the bottom row
        if idx >= len(axes) - ncols:
            ax.set_xlabel('Time relative to stimulus (s)', fontsize=12)
    
    # 7. Clean up empty subplots
    # If num_regions isn't a perfect multiple of ncols, hide the extra axes
    for j in range(idx + 1, len(axes)):
        fig8.delaxes(axes[j])
    
    # 8. Add a Single, Global Colorbar
    sm = ScalarMappable(cmap=cmap, norm=norm)
    sm.set_array([]) 
    # Pass the list of active axes to `ax` so the colorbar scales appropriately beside the grid
    cbar = fig8.colorbar(sm, ax=axes[:num_regions].tolist(), shrink=0.6, pad=0.03)
    cbar.set_label('Probe Depth (\u00B5m)', rotation=270, labelpad=15)
    fig8.suptitle('Mean LFP Waveforms Across Channels, Faceted by Region', fontsize=16, y=1.02)

    return fig8, axes

In [31]:
def calculate_and_plot(window, trial_type, triggers, processor, save_dir, mm, depth_map, mouse):
    PARAMS["window_lfp"] = window
    base_dir = Path(save_dir)
    mean_dir = base_dir / "data" / f"meanLFP{trial_type}"
    mean_path = mean_dir / "meanLFP.dat"
    if os.path.isfile(mean_path):
        print("Mean already computed, loading in as memmap")
        mean_clean, mean_meta = load_scaled_lfp_from_dat(str(mean_path))
        mean_sem = np.load(mean_dir / "mean_sem.npy")
    else:
        print(f"Step 4: Averaging {len(triggers)} Trials...")
        mean_clean, mean_sem = avg_lfp_by_trigger(mm, triggers, PARAMS, processor)
        save_scaled_lfp_to_dat(mean_clean, str(mean_dir / "meanLFP"), PARAMS['fs_lfp'])
        np.save(mean_dir / "mean_sem", mean_sem)
 
        
    # 5. VIRTUAL PROBE (AVERAGE)
    print("Step 5: Creating Virtual Probe (Average)...")
    virt_lfp_final, depth_axis = create_virtual_probe(mean_clean, depth_map)

    
    # 6. ANATOMY & CSD
    print("Step 6: Calculating CSD...")
    time_ax = np.linspace(*PARAMS['window_lfp'], mean_clean.shape[1])
    
    lfp_smooth = gaussian_filter(virt_lfp_final, sigma=[5, 0])
    csd = -1 * np.diff(lfp_smooth, n=2, axis=0)
    depth_csd = depth_axis[1:-1]


    #csd_dir = save_dir + r"\csd"
    #save_scaled_lfp_to_dat(csd, csd_dir)
    

    #Calculate psd
    signal_length = virt_lfp_final.shape[1]  # Assuming axis 1 is time; should equal 600
    
    # Dividing by 2 means a 300-sample window, allowing for roughly 3 overlapping segments
    nperseg_adjusted = signal_length // 2 
    
    # Calculate PSD with the adjusted parameters
    fs = PARAMS['fs_lfp']
    frequencies, psd = signal.welch(
        virt_lfp_final, 
        fs=fs, 
        window='hann', 
        nperseg=nperseg_adjusted,      
        noverlap=nperseg_adjusted // 2,  
        axis=1           
    )
    
    #Calculate MUA
    
    st = np.load(os.path.join(ks_dir, 'spike_times.npy')).flatten()
    sc = np.load(os.path.join(ks_dir, 'spike_clusters.npy')).flatten()
    sp = np.load(os.path.join(ks_dir, 'spike_positions.npy'))
    mua_map, t_ax, d_ax = calculate_spiking_mua_depth(st, sp, triggers, post_stim = window[1] * 1000)

    save_dir = base_dir / trial_type
    save_dir = str(save_dir)


    ccf_path = rf"Z:\Saij\ccf_channels\Mouse {mouse}.csv"
    has_ccf = os.path.isfile(ccf_path)

    if has_ccf:
        df = pd.read_csv(ccf_path)
        
        region_boundaries = df.groupby(['region_id', 'acronym'])['rel_y'].agg(
            min_rel_y='min',
            max_rel_y='max'
        ).reset_index()
        sorted_boundaries = region_boundaries.sort_values(by='min_rel_y', ascending=False)
        sorted_boundaries = sorted_boundaries.reset_index(drop = True)
        
        num_regions = len(sorted_boundaries)
        heights = sorted_boundaries['max_rel_y'] - sorted_boundaries['min_rel_y']
        bottoms = sorted_boundaries['min_rel_y']
        cmap = plt.get_cmap('tab20')
        colors = [cmap(i / num_regions) for i in range(num_regions)]
        
        
    
    #Plot avg
    fig3, ax2 = plt.subplots(figsize=(8, 6))
    vm_avg = np.percentile(np.abs(virt_lfp_final), 99)
    
    im2 = ax2.imshow(virt_lfp_final, aspect='auto', cmap='RdBu_r', origin='lower', 
                     vmin=-vm_avg, vmax=vm_avg,
                     extent=[time_ax[0], time_ax[-1], depth_axis[0], depth_axis[-1]])
    ax2.axvline(0, color='Black', lw=2, linestyle='--', label=f'Stimulus onset')
    if trial_type == "Loom":
        ax2.axvline(1.47, color='Black', lw=2, linestyle='--', label=f'Stimulus onset')
        ax2.axvline(2.94, color='Black', lw=2, linestyle='--', label=f'Stimulus onset')
    ax2.set_title("Averaged Evoked LFP")
    ax2.set_xlabel("Time from Stimulus (s)")
    ax2.set_ylabel("Distance from probe tip (µm)")
    ax2.legend(loc='upper right')
    fig3.colorbar(im2, ax=ax2, label="Voltage (µV)")
    save_pub_fig(fig3, "1_Avg_LFP", mouse, save_dir)

    if has_ccf and sorted_boundaries is not None:
        fig3_combo, (ax2_lfp, ax2_anatomy) = plt.subplots(
            nrows = 1,
            ncols = 2,
            figsize = (10,6),
            sharey = True,
            gridspec_kw = {"width_ratios": [4,1]}
        )


        im2_combo = ax2_lfp.imshow(
            virt_lfp_final, 
            aspect = "auto",
            cmap = "RdBu_r",
            origin = "lower", 
            vmin = -vm_avg,
            vmax = vm_avg,
            extent = [time_ax[0], time_ax[-1], depth_axis[0], depth_axis[-1]]
        )

        ax2_lfp.axvline(0, color = "Black", lw = 2, linestyle = "--", label = "Stimulus onset")
        if trial_type == "Loom":
            ax2_lfp.axvline(1.47, color = "Black", lw = 2, linestyle = "--", label = "Stimulus onset")
            ax2_lfp.axvline(2.94, color = "Black", lw = 2, linestyle = "--", label = "Stimulus onset")
        ax2_lfp.set_title("Average Evoked LFP")
        ax2_lfp.set_xlabel("Time from stimulus (s)")
        ax2_lfp.set_ylabel("Distance from probe tip (µm)")

        
        handles, labels = ax2_lfp.get_legend_handles_labels()
        by_label = dict(zip(labels, handles))
        ax2_lfp.legend(by_label.values(), by_label.keys(), loc='upper right')
        fig3_combo.colorbar(im2_combo, ax=ax2_lfp, label="Voltage (µV)")

            
        
        for i in range(num_regions):
            ax2_anatomy.bar(
                x=0, height=heights[i], bottom=bottoms[i], 
                color=colors[i], edgecolor='black', width=0.5
            )
            mid_point = bottoms[i] + (heights[i] / 2)
            ax2_anatomy.text(
                x=0, y=mid_point, s=sorted_boundaries['acronym'].iloc[i], 
                ha='center', va='center', color='black', fontweight='bold', fontsize=10
            )
            
        ax2_anatomy.set_title("Histology")
        ax2_anatomy.set_xticks([])
        ax2_anatomy.grid(axis='y', linestyle='--', alpha=0.7)
        
        # Save the combined plot and close it
        plt.tight_layout()
        save_pub_fig(fig3_combo, "1_Avg_LFP_with_Histology", mouse, save_dir)

    
    #CSD
    fig4, ax3 = plt.subplots(figsize=(8, 6))
    vm_csd = np.percentile(np.abs(csd), 99)
    
    im3 = ax3.imshow(csd, aspect='auto', cmap='viridis', origin='lower', 
                     vmin=-vm_csd, vmax=vm_csd,
                     extent=[time_ax[0], time_ax[-1], depth_csd[0], depth_csd[-1]])
    ax3.axvline(0, color='black', lw=2, linestyle='--', label=f'Stimulus onset')
    if trial_type == "Loom":
        ax3.axvline(1.47, color='Black', lw=2, linestyle='--', label=f'Stimulus onset')
        ax3.axvline(2.94, color='Black', lw=2, linestyle='--', label=f'Stimulus onset')
    ax3.set_title("Current Source Density (CSD)")
    ax3.set_xlabel("Time from Stimulus (s)")
    ax3.set_ylabel("Depth (µm)")
    ax3.legend(loc='upper right')
    fig4.colorbar(im3, ax=ax3, label="CSD (mV/mm²)")
    save_pub_fig(fig4, "2_CSD", mouse, save_dir)
    
    if has_ccf and sorted_boundaries is not None:
        fig4_combo, (ax3_csd, ax3_anatomy) = plt.subplots(
            nrows = 1,
            ncols = 2,
            figsize = (10,6),
            sharey = True,
            gridspec_kw = {"width_ratios": [4,1]}
        )


        im3_combo = ax3_csd.imshow(csd, aspect='auto', cmap='viridis', origin='lower', 
                     vmin=-vm_csd, vmax=vm_csd,
                     extent=[time_ax[0], time_ax[-1], depth_csd[0], depth_csd[-1]])

        ax3_csd.axvline(0, color = "Black", lw = 2, linestyle = "--", label = "Stimulus onset")
        if trial_type == "Loom":
            ax3_csd.axvline(1.47, color = "Black", lw = 2, linestyle = "--", label = "Stimulus onset")
            ax3_csd.axvline(2.94, color = "Black", lw = 2, linestyle = "--", label = "Stimulus onset")
        ax3_csd.set_title("Current Source Density (CSD)")
        ax3_csd.set_xlabel("Time from stimulus (s)")
        ax3_csd.set_ylabel("Distance from probe tip (µm)")

        
        handles, labels = ax3_csd.get_legend_handles_labels()
        by_label = dict(zip(labels, handles))
        ax3_csd.legend(by_label.values(), by_label.keys(), loc='upper right')
        fig4_combo.colorbar(im3_combo, ax=ax3_csd, label="CSD (mV/mm²)")

            
        
        for i in range(num_regions):
            ax3_anatomy.bar(
                x=0, height=heights[i], bottom=bottoms[i], 
                color=colors[i], edgecolor='black', width=0.5
            )
            mid_point = bottoms[i] + (heights[i] / 2)
            ax3_anatomy.text(
                x=0, y=mid_point, s=sorted_boundaries['acronym'].iloc[i], 
                ha='center', va='center', color='black', fontweight='bold', fontsize=10
            )
            
        ax3_anatomy.set_title("Histology")
        ax3_anatomy.set_xticks([])
        ax3_anatomy.grid(axis='y', linestyle='--', alpha=0.7)
        
        # Save the combined plot and close it
        plt.tight_layout()
        save_pub_fig(fig4_combo, "2_CSD_with_Histology", mouse, save_dir)

    #MUA
    
    fig6, ax5 = plt.subplots(figsize=(8, 6))
    
    vm_spiking = np.percentile(mua_map, 99.5) # Cap the maximum color at the 99.5th percentile to prevent blowout from massive individual units
    
    im5 = ax5.imshow(mua_map, aspect='auto', cmap='magma', origin='lower', 
                     vmin=0, vmax=vm_spiking,
                     extent=[t_ax[0], t_ax[-1], d_ax[0], d_ax[-1]])
    
    # Overlay the previously identified CSD sink for anatomical correlation
    ax5.axvline(0, color='white', lw=2, linestyle='--', label=f'Stimulus Onset')
    if trial_type == "Loom":
        ax5.axvline(1470, color='White', lw=2, linestyle='--', label=f'Stimulus onset')
        ax5.axvline(2940, color='White', lw=2, linestyle='--', label=f'Stimulus onset')
    ax5.set_title("Evoked Spiking Multi-Unit Activity (Spike Density)")
    ax5.set_xlabel("Time (ms)")
    ax5.set_ylabel("Probe Depth (µm)")
    ax5.legend(loc='upper right')
    fig6.colorbar(im5, ax=ax5, label="Firing Rate (Spikes/s)")
    
    save_pub_fig(fig6, "3_Spiking_MUA_Map", mouse, save_dir)
    
    if has_ccf and sorted_boundaries is not None:
        fig6_combo, (ax5_mua, ax5_anatomy) = plt.subplots(
            nrows = 1,
            ncols = 2,
            figsize = (10,6),
            sharey = True,
            gridspec_kw = {"width_ratios": [4,1]}
        )
        
        im5_combo = ax5_mua.imshow(mua_map, aspect='auto', cmap='magma', origin='lower', 
                     vmin=0, vmax=vm_spiking,
                     extent=[t_ax[0], t_ax[-1], d_ax[0], d_ax[-1]])
        
        ax5_mua.axvline(0, color='white', lw=2, linestyle='--', label=f'Stimulus Onset')
        if trial_type == "Loom":
            ax5_mua.axvline(1470, color='White', lw=2, linestyle='--', label=f'Stimulus onset')
            ax5_mua.axvline(2940, color='White', lw=2, linestyle='--', label=f'Stimulus onset')
        ax5_mua.set_title("Evoked Spiking Multi-Unit Activity (Spike Density)")
        ax5_mua.set_xlabel("Time (ms)")
        ax5_mua.set_ylabel("Probe Depth (µm)")
        handles, labels = ax5_mua.get_legend_handles_labels()
        by_label = dict(zip(labels, handles))
        ax5_mua.legend(by_label.values(), by_label.keys(), loc='upper right')
        fig6_combo.colorbar(im5_combo, ax=ax5_mua, label="Firing Rate (Spikes/s)")

        
        for i in range(num_regions):
            ax5_anatomy.bar(
                x=0, height=heights[i], bottom=bottoms[i], 
                color=colors[i], edgecolor='black', width=0.5
            )
            mid_point = bottoms[i] + (heights[i] / 2)
            ax5_anatomy.text(
                x=0, y=mid_point, s=sorted_boundaries['acronym'].iloc[i], 
                ha='center', va='center', color='black', fontweight='bold', fontsize=10
            )
            
        ax5_anatomy.set_title("Histology")
        ax5_anatomy.set_xticks([])
        ax5_anatomy.grid(axis='y', linestyle='--', alpha=0.7)
        
        # Save the combined plot and close it
        plt.tight_layout()
        save_pub_fig(fig6_combo, "3_Spiking_MUA_Map_with_Histology", mouse, save_dir)




    #PSD
    
    freq_mask = (frequencies >= 1) & (frequencies <= 100)
    freqs_plot = frequencies[freq_mask]
    psd_plot = psd[:, freq_mask]

    
    fig7, ax6 = plt.subplots(figsize=(8, 6))
    mesh = ax6.pcolormesh(
        freqs_plot, 
        depth_axis, 
        psd_plot, 
        shading='auto', 
        cmap='viridis',
        norm=LogNorm() # Log-scale normalization for power
    )
    

    ax6.set_xlabel('Frequency (Hz)')
    ax6.set_ylabel('Depth ($\mu m$)')
    ax6.set_title('Local Field Potential Power Spectral Density')

    cbar = fig7.colorbar(mesh, ax=ax6, label='Power ($V^2/Hz$)')
    save_pub_fig(fig7, "4_PSD", mouse, save_dir)
    
    if has_ccf and sorted_boundaries is not None:
        fig7_combo, (ax6_psd, ax6_anatomy) = plt.subplots(
            nrows = 1,
            ncols = 2,
            figsize = (10,6),
            sharey = True,
            gridspec_kw = {"width_ratios": [4,1]}
        )
        
        mesh_combo = ax6_psd.pcolormesh(
            freqs_plot, 
            depth_axis, 
            psd_plot, 
            shading='auto', 
            cmap='viridis',
            norm=LogNorm() # Log-scale normalization for power
        )
        
        ax6_psd.set_xlabel('Frequency (Hz)')
        ax6_psd.set_ylabel('Depth ($\mu m$)')
        ax6_psd.set_title('Local Field Potential Power Spectral Density')
    
        cbar_combo = fig7_combo.colorbar(mesh_combo, ax=ax6_psd, label='Power ($V^2/Hz$)')
        
        for i in range(num_regions):
            ax6_anatomy.bar(
                x=0, height=heights[i], bottom=bottoms[i], 
                color=colors[i], edgecolor='black', width=0.5
            )
            mid_point = bottoms[i] + (heights[i] / 2)
            ax6_anatomy.text(
                x=0, y=mid_point, s=sorted_boundaries['acronym'].iloc[i], 
                ha='center', va='center', color='black', fontweight='bold', fontsize=10
            )
            
        ax6_anatomy.set_title("Histology")
        ax6_anatomy.set_xticks([])
        ax6_anatomy.grid(axis='y', linestyle='--', alpha=0.7)
        
        # Save the combined plot and close it
        plt.tight_layout()
        save_pub_fig(fig7_combo, "4_PSD_with_Histology", mouse, save_dir)

        
    #Waveforms all channels
    num_channels, num_samples = mean_clean.shape
    time_vector = np.linspace(window[0], window[1], num_samples)
    depths = depth_map
    # 4. Set up the colormap based on depth
    # We normalize the depth values to a range of 0 to 1 for the colormap
    norm = Normalize(vmin=np.min(depths), vmax=np.max(depths))
    cmap = plt.get_cmap('coolwarm') # 'viridis', 'plasma', or 'coolwarm' work well
    
    # 5. Create the plot
    fig8, ax7 = plt.subplots(figsize=(8, 6))

    # 6. Loop through each channel and plot
    for i in range(num_channels):
        # Get the depth for this specific channel
        channel_depth = depths[i]
        
        # Get the specific color for this depth
        color = cmap(norm(channel_depth))
        
        # Plot the waveform
        ax7.plot(time_vector, mean_clean[i, :], color=color, alpha=0.7, linewidth=0.8)
    
    # 7. Add a vertical line at t=0 (stimulus onset)
    ax7.axvline(x=0, color='red', linestyle='--', label='Stimulus Onset')
    if trial_type == "Loom":
        ax7.axvline(x= 1.47, color='red', linestyle='--', label='Stimulus Onset')
        ax7.axvline(x= 2.94, color='red', linestyle='--', label='Stimulus Onset')
    # 8. Formatting the axes and labels
    ax7.set_title('Mean LFP Waveforms Across Channels', fontsize=14)
    ax7.set_xlabel('Time relative to stimulus (s)', fontsize=12)
    ax7.set_ylabel('Voltage (\u00B5V)', fontsize=12)
    ax7.set_xlim([window[0], window[1]])
    
    # 9. Add a colorbar to act as a depth legend
    # We create a ScalarMappable to link the colormap and the normalization
    sm = ScalarMappable(cmap=cmap, norm=norm)
    sm.set_array([]) # Required for older versions of matplotlib
    cbar = fig8.colorbar(sm, ax=ax7)
    cbar.set_label('Probe Depth (\u00B5m)', rotation=270, labelpad=15)
    save_pub_fig(fig8, "8_Waveforms", mouse, save_dir)


    #Virtual LFP waveforms by depth
    num_channels, num_samples = virt_lfp_final.shape
    time_vector = np.linspace(window[0], window[1], num_samples)
    depths = depth_axis
    # 4. Set up the colormap based on depth
    # We normalize the depth values to a range of 0 to 1 for the colormap
    norm = Normalize(vmin=np.min(depths), vmax=np.max(depth_axis))
    cmap = plt.get_cmap('coolwarm') # 'viridis', 'plasma', or 'coolwarm' work well
    
    # 5. Create the plot
    fig9, ax8 = plt.subplots(figsize=(8, 6))
    
    # 6. Loop through each channel and plot
    for i in range(num_channels):
        # Get the depth for this specific channel
        channel_depth = depths[i]
        
        # Get the specific color for this depth
        color = cmap(norm(channel_depth))
        
        # Plot the waveform
        ax8.plot(time_vector, virt_lfp_final[i, :], color=color, alpha=0.7, linewidth=0.8)
    
    # 7. Add a vertical line at t=0 (stimulus onset)
    ax8.axvline(x=0, color='red', linestyle='--', label='Stimulus Onset')
    if trial_type == "Loom":
        ax8.axvline(x= 1.47, color='red', linestyle='--', label='Stimulus Onset')
        ax8.axvline(x= 2.94, color='red', linestyle='--', label='Stimulus Onset')
    # 8. Formatting the axes and labels
    ax8.set_title('Mean LFP Waveforms Across Channels', fontsize=14)
    ax8.set_xlabel('Time relative to stimulus (s)', fontsize=12)
    ax8.set_ylabel('Voltage (\u00B5V)', fontsize=12)
    ax8.set_xlim([window[0], window[1]])
    
    # 9. Add a colorbar to act as a depth legend
    # We create a ScalarMappable to link the colormap and the normalization
    sm = ScalarMappable(cmap=cmap, norm=norm)
    sm.set_array([]) # Required for older versions of matplotlib
    cbar = fig9.colorbar(sm, ax=ax8)
    cbar.set_label('Probe Depth (\u00B5m)', rotation=270, labelpad=15)

    save_pub_fig(fig9, "9_Waveforms_virtual", mouse, save_dir)
        
    fig10, axes = plot_lfp_by_region(mean_clean, window, df, mouse, depth_map, trial_type)
    save_pub_fig(fig10, "10_Waveforms_by_region", mouse, save_dir)


In [32]:
def avg_and_plot_lfp_trials(recording_path, ks_dir, triggers, mouse,  stim_set, mice_stimset, save_dir = None,):
    depth_map = get_depth_map(ks_dir)
    #==============================
    #Initialise and set up recording and bad channel detection
    #==============================
    processor = NeuropixelsProcessor(384, PARAMS)
    mm = np.memmap(recording_path, dtype = "int16", mode = "r", 
                  shape = (os.path.getsize(recording_path)//(385*2), 385))

    calib_start = 30000 * 60 
    calib_chunk = mm[calib_start:calib_start+60000, :384].T
    calib_stats = processor.detect_bad_channels(calib_chunk)
    print(f"  > Flagged {np.sum(calib_stats['bad_mask'])} Bad Channels")
    
    print("Step 3: Extracting Diagnostic Sample...")
    # Grab 1 second of data near the first trigger
    samp_start = int(triggers[0, 0])
    samp_end = samp_start + 30000 
    samp_chunk = mm[samp_start:samp_end, :384].T

    
    sample_raw = processor.process_chunk(samp_chunk, clean=False)
    sample_clean = processor.process_chunk(samp_chunk, clean=True)
    
    sample_virtual, depth_axis_sample = create_virtual_probe(sample_clean, depth_map)

    #===============================
    #Handle trigger, logic, seperate all triggers, loom triggers and flash triggers
    #===============================
    stim = stim_set[mice_stimset[mouse]]
    
    stimulus_onset_triggers = []
    for trial in triggers:
        stimulus_start = trial[150]
        stimulus_onset_triggers.append(stimulus_start)

    flash_triggers = []
    for i, trial in enumerate(triggers):
        if stim[i] == 0:
            stimulus_start = trial[150]
            flash_triggers.append(stimulus_start)

    loom_triggers = []
    for i, trial in enumerate(triggers):
        if stim[i] == 1:
            loom1 = trial[150]
            loom2 = trial[172]
            loom3 = trial [173]
            loom_triggers.append(loom1)
            loom_triggers.append(loom2)
            loom_triggers.append(loom2)

    
    #=========================
    #Averaging logic, generally speaking need 3 avgs and concurrent plots done with different time frames
    #whole trial at (-0.1, 5) - avg over all triggers, not caring for loom or flash
    #Flash trials avged since flash is a stable block of stimulus just need to avg across trials for (-0.1, 5)
    #Loom trials, need to avg per loom i.e (-0.1, 0.5) across every loom, and avg per trial every third loom
    #By averaging at (-0.1, 5) for every third loom we effectively avg across trials and not within trials
    #To ensure correct logic can reshape loom_triggers to shape 20 x 3 and do loom_triggers[i, 0] to ensure first loom of every trial
    
    #important to denote loom onset on the plot above
    #need to generate all concurrent plots, CSD, PSD and MUA
    print("Calculating across all visual stimuli")
    calculate_and_plot((-0.1, 5), "All", stimulus_onset_triggers, processor, save_dir, mm, depth_map, mouse)
    print("Calculating across all Flashes")
    calculate_and_plot((-0.1, 5), "Flash", flash_triggers, processor, save_dir, mm, depth_map, mouse)
    loom_per_trial = loom_triggers[::3]
    print("Calculating looms across trials")
    calculate_and_plot((-0.1, 5), "Loom", loom_per_trial, processor, save_dir, mm, depth_map, mouse)
    print("Calculating across all looms")
    calculate_and_plot((-0.1, 0.7), "EveryLoom", loom_triggers, processor, save_dir, mm, depth_map, mouse)
    
    

        
    

In [33]:
for i in [2, 3, 4, 5, 6, 7, 8, 9]:
    n = str(i)
    print(f"Analysing Mouse {n}")
    recording_path = cfad.data["Neural"]['Extinction']['Concatenated Data']['Mouse ' + n] + r"\mouse" + n + "_extinction_concatenated_neural_data.dat"
    ks_dir = cfad.data["Neural"]['Extinction']['Concatenated Data']['kilosort4']['Mouse ' + n]
    triggers_p1_path = cfad.data["Neural"]['Extinction']['Triggers']['part 1']['Mouse ' + n]
    triggers_p2_path = cfad.data["Neural"]['Extinction']['Triggers']['part 2']['Mouse ' + n]
    mice_stimset = [None, 1, 1, 1, 2, 2, 1, 1, 2, 2]
    stim_set = [None, 
                [1, 0, 1, 0, 1, 0, 0, 0, 1, 0, 1, 1, 0, 1, 0, 0, 0, 1, 1, 1, 0, 0, 0, 1, 0, 1, 1, 1, 0, 0, 1, 1, 0, 0, 1, 0, 1, 1, 0, 1],
                [0, 1, 0, 1, 0, 1, 1, 1, 0, 1, 0, 0, 1, 0, 1, 1, 1, 0, 0, 0, 1, 1, 1, 0, 1, 0, 0, 0, 1, 1, 0, 0, 1, 1, 0, 1, 0 ,0 ,1 ,0]
               ]
    output_dir =r'Z:\Saij\Data\extinction\Mouse' + n
    
    triggers_p1 = np.array(scipy.io.loadmat(triggers_p1_path)["evt"]).flatten()
    triggers_p1 = triggers_p1[1:]
    triggers_p2 = np.array(scipy.io.loadmat(triggers_p2_path)["evt"]).flatten()
    triggers_p2 = triggers_p2[1:]
    triggers = np.append(triggers_p1, triggers_p2)
    triggers = triggers.reshape(40, 502)
    avg_and_plot_lfp_trials(recording_path, ks_dir, triggers, i, stim_set, mice_stimset, save_dir = output_dir)

Analysing Mouse 2
  > Flagged 5 Bad Channels
Step 3: Extracting Diagnostic Sample...
Calculating across all visual stimuli
Mean already computed, loading in as memmap
Step 5: Creating Virtual Probe (Average)...
Step 6: Calculating CSD...
Calculating depth-resolved Spike Density MUA...
MUA Spiking map generated.
Saved: Mouse2_1_Avg_LFP
Saved: Mouse2_1_Avg_LFP_with_Histology
Saved: Mouse2_2_CSD
Saved: Mouse2_2_CSD_with_Histology
Saved: Mouse2_3_Spiking_MUA_Map
Saved: Mouse2_3_Spiking_MUA_Map_with_Histology
Saved: Mouse2_4_PSD
Saved: Mouse2_4_PSD_with_Histology
Saved: Mouse2_8_Waveforms
Saved: Mouse2_9_Waveforms_virtual
Saved: Mouse2_10_Waveforms_by_region
Calculating across all Flashes
Mean already computed, loading in as memmap
Step 5: Creating Virtual Probe (Average)...
Step 6: Calculating CSD...
Calculating depth-resolved Spike Density MUA...
MUA Spiking map generated.
Saved: Mouse2_1_Avg_LFP
Saved: Mouse2_1_Avg_LFP_with_Histology
Saved: Mouse2_2_CSD
Saved: Mouse2_2_CSD_with_Histology

UnboundLocalError: local variable 'df' referenced before assignment

In [ ]:
for i in [2, 3, 4, 5, 6, 7, 8, 9]:
    n = str(i)
    print(f"Analysing Mouse {n}")
    recording_path = cfad.data["Neural"]['Renewal']['Concatenated Data']['Mouse ' + n] + r"\mouse" + n + "_renewal_concatenated_neural_data.dat"
    ks_dir = cfad.data["Neural"]['Renewal']['Concatenated Data']['kilosort4']['Mouse ' + n]
    triggers_p1_path = cfad.data["Neural"]['Renewal']['Triggers']['part 1']['Mouse ' + n]
    triggers_p2_path = cfad.data["Neural"]['Renewal']['Triggers']['part 2']['Mouse ' + n]
    mice_stimset = [None, 1, 1, 1, 2, 2, 1, 1, 2, 2]
    stim_set = [None, 
                [1, 0, 1, 0, 1, 0, 0, 0, 1, 0, 1, 1, 0, 1, 0, 0, 0, 1, 1, 1, 0, 0, 0, 1, 0, 1, 1, 1, 0, 0, 1, 1, 0, 0, 1, 0, 1, 1, 0, 1],
                [0, 1, 0, 1, 0, 1, 1, 1, 0, 1, 0, 0, 1, 0, 1, 1, 1, 0, 0, 0, 1, 1, 1, 0, 1, 0, 0, 0, 1, 1, 0, 0, 1, 1, 0, 1, 0 ,0 ,1 ,0]
               ]
    output_dir =r'Z:\Saij\Data\renewal\Mouse' + n
    
    triggers_p1 = np.array(scipy.io.loadmat(triggers_p1_path)["evt"]).flatten()
    triggers_p1 = triggers_p1[1:]
    triggers_p2 = np.array(scipy.io.loadmat(triggers_p2_path)["evt"]).flatten()
    triggers_p2 = triggers_p2[1:]
    triggers = np.append(triggers_p1, triggers_p2)
    triggers = triggers.reshape(40, 502)
    avg_and_plot_lfp_trials(recording_path, ks_dir, triggers, i, stim_set, mice_stimset, save_dir = output_dir)
